In [ ]:
using Bessels
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/config/requirements.jl")
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/utils/FFT_perso.jl");

In [ ]:
const ∞ = Inf
ħ=1
e=1
β = ∞ # Case zero temperature T=0, β → ∞
ϕ_quantum=1/2

#### Caracterization of the circuit #####

ω_a = 3.8533*2*pi*(rand(1)[1]/5+0.9)
ω_b = 2*ω_a

φ_a = 0.11
Z_a = 2*ϕ_quantum^2*φ_a^2
C_a_inv = Z_a*ω_a
E_Ca = C_a_inv/2

φ_b = 0.204
Z_b = 2*ϕ_quantum^2*φ_b^2
C_b_inv = Z_b*ω_b
E_Cb = C_b_inv/2

E_J = 37.12*2*pi*1.5


##Drive
ω_d=ω_b
ω_p=2ω_a-ω_d

ϵ_0 = 0.8#4581818181818182
g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b # (Leon's paper)
ϵ_d=10.12*g2

buffer_phase = 0.0
pump_phase = 0.0;

##Env

κ_b=1/10.4; # 1/ns from Leon's paper
κ_2 = 4*g2^2/κ_b
κ_a = 0#3e-3*κ_2 #from Leon's paper

In [ ]:
ω_a

In [ ]:
t_max = 40/κ_b
t_min = 0.0

re_a_0, im_a_0 = 2, 2
ϕ_a_0, q_a_0=sqrt(2*Z_a)*re_a_0, sqrt(2/Z_a)*im_a_0
ϕ_b_0, q_b_0=0.0, 0.0

system = SystemATS(ω_a, ω_b, E_J, E_Ca, E_Cb, ϕ_a_0, q_a_0, ϕ_b_0, q_b_0)
drive = DriveTwoModesBasics(ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase)
environment = MeanField(κ_a, κ_b)

tspan = (t_min, t_max) # Time span for the simulation

#preparation
system_prepared = prepare(system)
drive_prepared = prepare(drive)
environment_prepared = prepare(system,environment)

dt = get_dt(system, environment)
dt_save=2pi/ω_a
t_save = t_max-30/κ_b:dt_save:t_max 
u_0_tot = get_init(system_prepared, environment_prepared)

p = SimulationParams(system_prepared, drive_prepared, environment_prepared)
    
ϕ_CL_a, ϕ_CL_b, q_CL_a, q_CL_b, t_list_CL = default_solver(u_0_tot, tspan, p, dt; equation=default_equation!, t_save = t_save)

In [ ]:
p1,p2 = plot_mode(ϕ_CL_a[1:end], q_CL_a[1:end], t_list_CL[1:end], ω_a, Z_a)

plot_perso(p1,p2,layout=(1,2),
xlabel=L"\Re(a)", ylabel=L"\Im(a)",
title=["Resonator a" "Resonator a RF"],
legend=false)

In [ ]:
t_max=20/κ_b
a_0 = 1.5 + 1im
b_0 = 0 + 0im

system = SystemATS_ladder(ω_a, ω_b, E_J, E_Ca, E_Cb, a_0, b_0)
drive = DriveTwoModesJA(ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase)
environment = MeanField(κ_a, κ_b)
a, b, t_list = simulator(t_max, system, drive, environment);
a_rot = a.*exp.(1im*ω_a*t_list);
b_rot = b.*exp.(1im*ω_b*t_list);

In [ ]:
plot(real.(a_rot[1:end]),imag.(a_rot[1:end]))

In [ ]:
plot(t_list*κ_b, φ_a*2*real.(a_rot)+φ_b*2*real.(b_rot), label="")
hline!([-pi-0.5],label="")
hline!([pi-0.5], label="")
plot!(xlims=(0,7))

In [ ]:
#ATS + Mean_Field + ladder_operator
function test_equation!(du,u,p::SimulationParams{SystemATS_ladder_prepared, D, MeanField}, t) where D<:DriveTwoModes_prepared
    ω_a, ω_b, E_J, Z_a, Z_b, ϕ_quantum = p.system_prepared.system.ω_a, p.system_prepared.system.ω_b, p.system_prepared.system.E_J, p.system_prepared.Z_a, p.system_prepared.Z_b, p.system_prepared.ϕ_quantum
    ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase =  p.drive_prepared.drive.ϵ_d, p.drive_prepared.drive.ω_d, p.drive_prepared.drive.buffer_phase, p.drive_prepared.drive.ϵ_0, p.drive_prepared.drive.ω_p, p.drive_prepared.drive.pump_phase

    a = u[1]
    b = u[2]

    coupling = -4*besselj(2,pi)*((a+conj(a))/sqrt(2) * sqrt(Z_a)/ϕ_quantum/pi + (b+conj(b))/sqrt(2) * sqrt(Z_b)/ϕ_quantum/pi)^2 / ϕ_quantum / sqrt(2)
    sin_pump = 2*E_J*p.drive_prepared.drive_pump(ϵ_0, ω_p, t, pump_phase)
        
    du[1] = -1im*(ω_a*a - sqrt(Z_a)*sin_pump*coupling) - p.environment_prepared.κ_a * a
    du[2] = -1im*(ω_b*b + p.drive_prepared.drive_buffer(ϵ_d, ω_d, t, buffer_phase) - sqrt(Z_b)*sin_pump*coupling) - p.environment_prepared.κ_b * b
    return nothing
end


In [ ]:
t_max=20/κ_b
a_0 = 1.5 + 1im
b_0 = 0 + 0im

system = SystemATS_ladder(ω_a, ω_b, E_J, E_Ca, E_Cb, a_0, b_0)
drive = DriveTwoModesJA(ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase)
environment = MeanField(κ_a, κ_b)
a, b, t_list = simulator(t_max, system, drive, environment, equation = test_equation!);
a_rot = a.*exp.(1im*ω_a*t_list);
b_rot = b.*exp.(1im*ω_b*t_list);

In [ ]:
plot(real.(a_rot[1:end]),imag.(a_rot[1:end]))

In [ ]:
x=-1:0.01:1
y_sin_C = @. 2*((besselj(1,pi)+3*besselj(3,pi))*x-4*besselj(3,pi)*x^3)
y_sin = @. sin(pi*x)

y_cos = @. cos(pi*x)
y_cos_C = @. besselj(0, pi) + 2*besselj(2, pi) - 4*besselj(2,pi)*x^2;

In [ ]:
plot(x, y_sin)
plot!(x, y_sin_C)
plot!(x, y_cos)
plot!(x, y_cos_C)

In [ ]:
plot(x, cos.(pi*x .+0.5))
plot!(x, cos(0.5)*y_cos_C - sin(0.5)*y_sin_C)